<a href="https://colab.research.google.com/github/akashkguw/orion/blob/akashkg/orion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Orion: Sparse Attention for Long-Context Transformers

This notebook demonstrates training decoder-only Transformers with sparse attention (window + expander edges) for efficient long-context modeling.

**Key Features:**
- Sparse Attention - O(T*(W+d)) complexity vs O(T^2) for dense (7x faster on 512 tokens)
- Reproducible - Deterministic training with seed control
- Configurable - YAML-based configs for easy experimentation
- Well-tested - 82+ tests covering all components

## Setup: Install Dependencies

In [ ]:
import os
import shutil

os.chdir("/")
os.makedirs("/content", exist_ok=True)
os.chdir("/content")
if os.path.exists("orion"):
    shutil.rmtree("orion")

In [ ]:
!git clone https://github.com/akashkguw/orion.git
%cd /content/orion
!ls
!pip -q install -r requirements.txt -r requirements-dev.txt

Cloning into 'orion'...
remote: Enumerating objects: 316, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 316 (delta 98), reused 98 (delta 69), pack-reused 175 (from 1)
Receiving objects: 100% (316/316), 104.53 KiB | 240.00 KiB/s, done.
Resolving deltas: 100% (159/159), done.
/content/orion
configs   orion.ipynb		requirements.txt
LICENSE   pyproject.toml	SPARSE_ATTENTION_ARCHITECTURE.md
Makefile  README.md		tests
orion	  requirements-dev.txt


In [ ]:
import importlib.util

print("torch installed:", importlib.util.find_spec("torch") is not None)

torch installed: True


In [ ]:
# If Torch is not available, uncomment and run:
# !pip -q install torch

## What is Sparse Attention?

Sparse attention reduces computational complexity from O(T^2) to O(T*(W+d)) by attending to only a subset of positions:

- **Local Window** - Dense context for nearby tokens (short-range dependencies)
- **Expander Edges** - Structured long-range connections using modular arithmetic

### Example Pattern

For query at position q=10 with window_size=4, expander_degree=3:

```
Window:   [7, 8, 9, 10]           (local context)
Expander: [9, 6, 1]               (long-range via quadratic residues)
Union:    [1, 6, 7, 8, 9, 10]     (7 positions vs 11 for dense)
```

### Performance

| Metric | Dense | Sparse | Speedup |
|--------|-------|--------|----------|
| Complexity | O(T^2) | O(T*(W+d)) | ~7x |
| Memory | O(T^2*Dh) | O(T*(W+d)*Dh) | ~7x |
| Example (T=512, W=64, d=8) | 262K pos/query | 36.8K pos/query | 7.1x |

## Quick Start: Train with Dense Attention

### Execute Training (Smoke Test - 20 steps)

In [ ]:
!python -m orion.train --config configs/golden.yaml

{'step': 1, 'loss': 5.717885971069336, 'ppl': 304.2610168457031, 'vram_max_mb': 42}
{'step': 2, 'loss': 5.689811706542969, 'ppl': 295.8379211425781, 'vram_max_mb': 52}
{'step': 3, 'loss': 5.715023994445801, 'ppl': 303.3914794921875, 'vram_max_mb': 52}
{'step': 4, 'loss': 5.690974235534668, 'ppl': 296.1820373535156, 'vram_max_mb': 52}
{'step': 5, 'loss': 5.726962089538574, 'ppl': 307.03509521484375, 'vram_max_mb': 52}
{'step': 6, 'loss': 5.705636978149414, 'ppl': 300.5568542480469, 'vram_max_mb': 52}
{'step': 7, 'loss': 5.694186210632324, 'ppl': 297.1348876953125, 'vram_max_mb': 52}
{'step': 8, 'loss': 5.713808059692383, 'ppl': 303.0227966308594, 'vram_max_mb': 52}
{'step': 9, 'loss': 5.678920745849609, 'ppl': 292.6334228515625, 'vram_max_mb': 52}
{'step': 10, 'loss': 5.721549987792969, 'ppl': 305.3778991699219, 'vram_max_mb': 52}
{'step': 11, 'loss': 5.731828689575195, 'ppl': 308.532958984375, 'vram_max_mb': 52}
{'step': 12, 'loss': 5.722729206085205, 'ppl': 305.7381896972656, 'vram_ma

### Evaluate Model

In [ ]:
!python -m orion.eval --config configs/golden.yaml --checkpoint runs/latest/checkpoint.pt

{'loss': 5.6983537673950195, 'ppl': 298.37579345703125}


### View Training Logs

In [ ]:
!tail -n 5 runs/latest/metrics.jsonl

{"step": 16, "loss": 5.6980156898498535, "ppl": 298.2749328613281, "vram_max_mb": 52, "wall_time_s": 0.9006004333496094}
{"step": 17, "loss": 5.688562393188477, "ppl": 295.4685363769531, "vram_max_mb": 52, "wall_time_s": 0.9065103530883789}
{"step": 18, "loss": 5.720730304718018, "ppl": 305.127685546875, "vram_max_mb": 52, "wall_time_s": 0.9124374389648438}
{"step": 19, "loss": 5.696990013122559, "ppl": 297.96917724609375, "vram_max_mb": 52, "wall_time_s": 0.9183323383331299}
{"step": 20, "loss": 5.673758506774902, "ppl": 291.1266784667969, "vram_max_mb": 52, "wall_time_s": 0.924290657043457}


## Train with Sparse Attention

### Sparse Attention Configuration

Sparse attention is configured via YAML:

```yaml
attention:
  backend: sparse
  window_size: 64        # Local window size
  expander_degree: 8     # Long-range neighbors
  sparse_impl: auto      # auto | gather | flex
  sparse_block_size: 128 # Block size for flex backend
```

**When to use sparse attention:**
- Long sequences (512+)
- Limited GPU memory
- Need stable training with diverse patterns
- Short sequences (<256) - dense is simpler

### Sparse Attention Smoke Test (5 steps)

In [ ]:
!python -m orion.train --config configs/exp_orion_sparse_smoke.yaml

{'step': 1, 'loss': 5.720929145812988, 'ppl': 305.1883544921875, 'vram_max_mb': 97}
{'step': 2, 'loss': 5.702025413513184, 'ppl': 299.4733581542969, 'vram_max_mb': 105}
{'step': 3, 'loss': 5.702598571777344, 'ppl': 299.64501953125, 'vram_max_mb': 105}
{'step': 4, 'loss': 5.699774265289307, 'ppl': 298.7999572753906, 'vram_max_mb': 105}
{'step': 5, 'loss': 5.720228672027588, 'ppl': 304.9746398925781, 'vram_max_mb': 105}


### Evaluate Sparse Attention Model

In [ ]:
!python -m orion.eval --config configs/exp_orion_sparse_smoke.yaml --checkpoint runs/exp_orion_sparse_smoke/checkpoint.pt

{'loss': 5.701145172119141, 'ppl': 299.2098388671875}


## Train with Dense Attention (Full Dataset)

### Dense Attention on Shakespeare Dataset

Uses standard dense attention (O(T^2)) for comparison. Good for understanding baseline performance.

In [ ]:
!python -m orion.train --config configs/tinyshakespeare_dense.yaml

{'step': 10, 'loss': 3.113189220428467, 'ppl': 22.492664337158203, 'vram_max_mb': 352}
{'step': 20, 'loss': 2.925708293914795, 'ppl': 18.647430419921875, 'vram_max_mb': 352}
{'step': 30, 'loss': 2.813502073287964, 'ppl': 16.668190002441406, 'vram_max_mb': 352}
{'step': 40, 'loss': 2.6772241592407227, 'ppl': 14.54466438293457, 'vram_max_mb': 352}
{'step': 50, 'loss': 2.636259078979492, 'ppl': 13.960880279541016, 'vram_max_mb': 352}
{'step': 60, 'loss': 2.6029791831970215, 'ppl': 13.50390911102295, 'vram_max_mb': 352}
{'step': 70, 'loss': 2.561068534851074, 'ppl': 12.949647903442383, 'vram_max_mb': 352}
{'step': 80, 'loss': 2.547773599624634, 'ppl': 12.7786226272583, 'vram_max_mb': 352}
{'step': 90, 'loss': 2.5268261432647705, 'ppl': 12.513726234436035, 'vram_max_mb': 352}
{'step': 100, 'loss': 2.5043938159942627, 'ppl': 12.236140251159668, 'vram_max_mb': 352}
{'step': 110, 'loss': 2.53663969039917, 'ppl': 12.63713550567627, 'vram_max_mb': 352}
{'step': 120, 'loss': 2.5172715187072754, '

## Train with Sparse Attention (Full Dataset)

### Sparse Attention on Shakespeare Dataset

Uses sparse attention (O(T*(W+d))) for efficient long-context modeling. Compare performance with dense attention above.

In [ ]:
!python -m orion.train --config configs/tinyshakespeare_sparse.yaml

{'step': 10, 'loss': 3.2056589126586914, 'ppl': 24.671751022338867, 'vram_max_mb': 368}
{'step': 20, 'loss': 2.921800136566162, 'ppl': 18.57469367980957, 'vram_max_mb': 368}
{'step': 30, 'loss': 2.788296699523926, 'ppl': 16.253311157226562, 'vram_max_mb': 368}
{'step': 40, 'loss': 2.7308874130249023, 'ppl': 15.3464994430542, 'vram_max_mb': 368}
{'step': 50, 'loss': 2.6702377796173096, 'ppl': 14.443403244018555, 'vram_max_mb': 368}
{'step': 60, 'loss': 2.6138367652893066, 'ppl': 13.651327133178711, 'vram_max_mb': 368}
{'step': 70, 'loss': 2.5466480255126953, 'ppl': 12.764246940612793, 'vram_max_mb': 368}
{'step': 80, 'loss': 2.546884536743164, 'ppl': 12.767266273498535, 'vram_max_mb': 368}
{'step': 90, 'loss': 2.5422606468200684, 'ppl': 12.708368301391602, 'vram_max_mb': 368}
{'step': 100, 'loss': 2.556858777999878, 'ppl': 12.895246505737305, 'vram_max_mb': 368}
{'step': 110, 'loss': 2.5199859142303467, 'ppl': 12.428421974182129, 'vram_max_mb': 368}
{'step': 120, 'loss': 2.5261125564575

## Comparison: Dense vs Sparse Attention

### Available Configurations

| Config | Attention | Window | Expander | Use Case |
|--------|-----------|--------|----------|----------|
| golden.yaml | Dense | - | - | Quick smoke test |
| tinyshakespeare_dense.yaml | Dense | - | - | Baseline comparison |
| exp_orion_sparse_smoke.yaml | Sparse | 64 | 8 | Quick sparse test |
| tinyshakespeare_sparse.yaml | Sparse | 64 | 8 | Full sparse training |
| exp_sparse_window_256.yaml | Sparse | 256 | 8 | Larger window |

### Performance Metrics

After training, compare:
- Loss - Lower is better
- Perplexity - Lower is better
- Training Time - Sparse should be faster
- Memory Usage - Sparse should use less VRAM

## Comprehensive Metrics Logging

Orion logs comprehensive P0 metrics to `runs/latest/metrics.jsonl` in JSONL format:

### Step Metrics (every step)
- `loss`, `ppl` - Training loss and perplexity
- `throughput_tokens_per_sec` - Training throughput
- `grad_norm`, `grad_clipped` - Gradient statistics
- `step_time_ms` - Time per step
- `accuracy_top1` - Next-token prediction accuracy
- `learning_rate` - Current learning rate

### Window Metrics (every 50 steps)
- `vram_peak_mib` - Peak GPU memory usage
- `divergence_rate` - Fraction of diverged steps
- `activation_norm_rms` - RMS of residual stream
- `attention_entropy`, `attention_entropy_normalized` - Attention distribution entropy
- `clip_rate` - Fraction of clipped gradient steps
- `valid_neighbor_fraction` - (Sparse only) Effective degree / degree
- `attention_mass_window_pct`, `attention_mass_expander_pct` - (Sparse only) Mass split

### Run Metrics (once per run)
- `attention_degree` - Total attention neighbors (W + d)
- `compute_proxy_per_token`, `compute_proxy_per_seq`, `compute_proxy_per_step` - Complexity estimates

### Eval Metrics (every 1000 steps)
- `eval_ppl_512`, `eval_ppl_1024`, `eval_ppl_2048`, `eval_ppl_4096` - Long-context evaluation

View metrics with:
```bash
cat runs/latest/metrics.jsonl | jq .
```

## Advanced: Custom Configuration

### Create Custom Sparse Attention Config

You can create custom configs by modifying YAML files. Example:

```yaml
run:
  out_dir: runs/my_experiment
  seed: 42
  steps: 500
  log_every: 10
  save_every: 100

data:
  dataset: tinyshakespeare
  seq_len: 512          # Longer sequences benefit from sparse
  batch_size: 8

model:
  name: orion_decoder   # Use orion_decoder for custom attention backends
  d_model: 256
  n_layers: 4
  n_heads: 4

attention:
  backend: sparse       # Options: dense, sparse, window
  window_size: 128      # Larger window for more local context
  expander_degree: 16   # More long-range neighbors
  sparse_impl: auto     # auto | gather | flex
  sparse_block_size: 128

optim:
  lr: 3e-4
```

## Resources and Documentation

- README: Full documentation with examples
- SPARSE_ATTENTION_ARCHITECTURE.md: Deep technical dive into sparse attention
- GitHub: https://github.com/akashkguw/orion

### Key Files

- orion/attention/sparse.py - Sparse attention implementation
- orion/attention/dense.py - Dense attention implementation
- configs/ - Training configurations
- tests/test_sparse_attention.py - 82+ tests for sparse attention

## Troubleshooting

### Out of Memory (OOM)
- Use sparse attention instead of dense
- Reduce batch_size in config
- Reduce seq_len in config
- Reduce d_model in config

### Slow Training
- Check GPU is being used: !nvidia-smi
- Use sparse attention for long sequences
- Increase batch_size if memory allows

### Training Not Converging
- Try different seed values
- Adjust learning rate (lr in config)
- Increase training steps (steps in config)
- Check data loading with !head -c 100 data/tinyshakespeare.txt